# LeetCode 3888 — Minimum Operations to Make All Grid Elements Equal bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-make-all-grid-elements-equal/
- Độ khó: **Hard** · 🔒 **Premium**
- Chủ đề: Ma trận, tham lam quét từ góc trên-trái, mảng hiệu / cửa sổ trượt 2D, tham số hóa theo giá trị đích `T` (hàm affine), bất biến tổng

> **Ghi chú Premium:** đề bài được viết lại theo hiểu biết về bài (dựa trên bản mô tả công khai của doocs). Nếu có tài khoản Premium, nên đối chiếu lại đề gốc và skeleton.
>
> **Ghi chú quan trọng:** lời giải tham khảo của doocs chỉ thử hai giá trị đích `T = max` và `T = max + 1`. Notebook này chỉ ra **phản ví dụ** (`[[5,0,5],[5,0,5]], k = 2` có đáp án `10` với `T = 10`) và trình bày lời giải đúng cho mọi `T`. Nên đối chiếu lại với bộ test chính thức.

Skeleton (theo doocs):

```python
class Solution:
    def minOperations(self, grid: List[List[int]], k: int) -> int:
```

## 1. Mục tiêu học

1. Hiểu lập luận **tham lam quét từ góc trên-trái**: ô `(i, j)` chỉ còn có thể được tăng bởi phép toán **đặt góc tại chính nó**, nên số lần đặt tại `(i, j)` bị **ép buộc**.
2. Cài đặt việc cộng `k × k` trong `O(1)` mỗi lần bằng **mảng hiệu 2D** hoặc **cửa sổ trượt theo cột**.
3. Nhận ra mọi đại lượng là **hàm affine của giá trị đích `T`** — từ đó giải ra `T` (có thể là duy nhất và lớn hơn `max + 1` rất nhiều!).
4. Dùng bất biến tổng: số phép toán `= (m·n·T − Σgrid) / k²`.
5. Kiểm thử với vét cạn số lần đặt phép toán trên lưới nhỏ, và phát hiện chỗ sai của cách "chỉ thử `max` và `max + 1`".

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho lưới `grid` kích thước `m × n` và số `k`. Mỗi thao tác: chọn một **ma trận con `k × k`** (nằm trọn trong lưới) và **tăng 1** mọi ô trong đó. Tìm số thao tác ít nhất để mọi ô **bằng nhau**; nếu không thể, trả về `-1`.

**Ví dụ 1:** `grid = [[3,3,5],[3,3,5]], k = 2` → `2` (tăng ma trận con hai cột trái hai lần).

**Ví dụ 2:** `grid = [[1,2],[2,3]], k = 1` → `4` (mỗi ô tăng lên 3: `2 + 1 + 1 + 0`).

**Ràng buộc:** `1 ≤ m, n ≤ 1000`, `-10^5 ≤ grid[i][j] ≤ 10^5`, `1 ≤ k ≤ min(m, n)`.

In [1]:
from typing import List
from itertools import product
import random, time

examples = [([[3, 3, 5], [3, 3, 5]], 2, 2), ([[1, 2], [2, 3]], 1, 4)]
for g, k, exp in examples:
    print(g, "k =", k, "-> kỳ vọng", exp)

[[3, 3, 5], [3, 3, 5]] k = 2 -> kỳ vọng 2
[[1, 2], [2, 3]] k = 1 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Mảng hiệu 2D

Muốn cộng `v` vào hình chữ nhật `[r1, r2) × [c1, c2)`: `D[r1][c1] += v`, `D[r2][c1] -= v`, `D[r1][c2] -= v`, `D[r2][c2] += v`. Giá trị thật tại `(i, j)` là **tổng tiền tố 2D** của `D` tới `(i, j)`. Có thể tính tiền tố "trực tiếp" khi quét theo hàng: `P[i][j] = D[i][j] + P[i-1][j] + P[i][j-1] - P[i-1][j-1]`.

In [2]:
m, n = 3, 4
D = [[0] * (n + 1) for _ in range(m + 1)]
def add_rect(r1, c1, r2, c2, v):
    D[r1][c1] += v; D[r2][c1] -= v; D[r1][c2] -= v; D[r2][c2] += v
add_rect(0, 1, 2, 3, 5)
P = [[0] * (n + 1) for _ in range(m + 1)]
for i in range(m):
    for j in range(n):
        P[i][j] = D[i][j] + (P[i - 1][j] if i else 0) + (P[i][j - 1] if j else 0) - (P[i - 1][j - 1] if i and j else 0)
for i in range(m):
    print(P[i][:n])

[0, 5, 5, 0]
[0, 5, 5, 0]
[0, 0, 0, 0]


### 3.2. Cửa sổ trượt theo cột (cách cộng `k × k` gọn hơn)

Giữ `col[c]` = tổng số lần đặt phép toán tại cột `c` ở các hàng `(i-k, i]` (những phép còn phủ hàng `i`). Khi đi dọc hàng `i`, cửa sổ `window = Σ col[c]` với `c ∈ (j-k, j]` chính là lượng đã cộng vào ô `(i, j)`. Rời hàng thì trừ đi các phép đặt ở hàng `i-k+1`… (hàng đã ra khỏi phạm vi).

In [3]:
def coverage_by_window(m, n, k, ops):
    """ops[i][j] = số lần đặt góc tại (i, j). Trả về lượng cộng vào từng ô."""
    col = [0] * n
    out = [[0] * n for _ in range(m)]
    for i in range(m):
        if i >= k:
            for c in range(n):
                col[c] -= ops[i - k][c]
        for c in range(n):
            col[c] += ops[i][c]
        window = 0
        for j in range(n):
            window += col[j]
            if j >= k:
                window -= col[j - k]
            out[i][j] = window
    return out

ops = [[1, 0, 0], [0, 0, 2], [0, 0, 0]]
for row in coverage_by_window(3, 4, 2, [r + [0] for r in ops]):
    print(row)

[1, 1, 0, 0]
[1, 1, 2, 2]
[0, 0, 2, 2]


### 3.3. Hàm affine theo một ẩn

Một đại lượng có dạng `a + b·T` lưu bằng cặp `(a, b)`. Cộng/trừ cặp từng thành phần. Phương trình `a + b·T = 0`: nếu `b ≠ 0` thì `T = -a / b` (cần là số nguyên); nếu `b = 0` thì cần `a = 0` (đúng với mọi `T`) hoặc vô nghiệm.

In [4]:
def solve_affine(a, b):
    if b == 0:
        return "mọi T" if a == 0 else "vô nghiệm"
    return -a / b

print(solve_affine(-10, 1), solve_affine(0, 0), solve_affine(3, 0), solve_affine(7, -2))

10.0 mọi T vô nghiệm 3.5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Phép toán chỉ **tăng** ⇒ cuối cùng mọi ô bằng một giá trị `T ≥ max(grid)`.

Quét ô theo thứ tự hàng (trái → phải, trên → dưới). Ô `(i, j)` có thể được phủ bởi phép đặt góc tại `(r, c)` với `r ≤ i`, `c ≤ j`. Các phép đặt góc **sau** `(i, j)` theo thứ tự quét (hàng dưới, hoặc cùng hàng nhưng bên phải) **không** phủ `(i, j)`, **trừ** phép đặt góc tại chính `(i, j)`. Vậy khi tới `(i, j)`:

- nếu đặt góc tại `(i, j)` được (`i ≤ m-k`, `j ≤ n-k`): số lần đặt **bắt buộc** là `T - giá_trị_hiện_tại` (phải `≥ 0`);
- nếu không đặt được: giá trị hiện tại **phải bằng** `T`.

Tính tay **phản ví dụ** `grid = [[5,0,5],[5,0,5]]`, `k = 2`: chỉ có hai vị trí đặt `(0,0)` và `(0,1)`.

- Cột 0: `5 + x₀ = T`; cột 1: `0 + x₀ + x₁ = T`; cột 2: `5 + x₁ = T` ⇒ `x₀ = x₁ = T - 5` và `2(T - 5) = T` ⇒ **`T = 10`**, tổng `10` phép.
- Thử `T = 5` hay `T = 6` (như doocs) đều thất bại — đáp án đúng là `10`, không phải `-1`.

## 5. Cách 1 — Brute Force: vét cạn số lần đặt tại mỗi vị trí

Có `(m-k+1)(n-k+1)` vị trí đặt góc. Thử mọi vector số lần `x ∈ [0, B]^{vị trí}`, tính lưới cuối, kiểm tra mọi ô bằng nhau, lấy tổng nhỏ nhất. Chỉ dùng cho lưới tí hon.

In [5]:
def brute(grid, k, B=12):
    m, n = len(grid), len(grid[0])
    pos = [(i, j) for i in range(m - k + 1) for j in range(n - k + 1)]
    best = None
    for xs in product(range(B + 1), repeat=len(pos)):
        s = sum(xs)
        if best is not None and s >= best:
            continue
        g = [row[:] for row in grid]
        for (i, j), x in zip(pos, xs):
            if x:
                for a in range(i, i + k):
                    for b in range(j, j + k):
                        g[a][b] += x
        if len({v for row in g for v in row}) == 1:
            best = s
    return -1 if best is None else best

for g, k, exp in examples:
    print(brute(g, k, B=4), "kỳ vọng", exp)
print("phản ví dụ [[5,0,5],[5,0,5]], k=2:", brute([[5, 0, 5], [5, 0, 5]], 2, B=6))

2 kỳ vọng 2
4 kỳ vọng 4
phản ví dụ [[5,0,5],[5,0,5]], k=2: 10


**Nút thắt:** `(B+1)^{số vị trí}`; còn chưa biết chặn `B` thế nào. Mục 4 cho thấy khi biết `T`, mọi `x` bị **ép buộc** ⇒ chỉ cần tìm đúng `T`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Với `T` cố định, kiểm tra trong `O(mn)`** (hàm `check(T)`): quét tham lam như mục 4, dùng cửa sổ trượt/mảng hiệu để biết lượng đã cộng. Trả về tổng số phép hoặc `-1`.

**Bước 2 — Mọi thứ là affine theo `T`.** Chạy tham lam với `T` là **ẩn**: giá trị hiện tại, số lần đặt `x_{ij}` đều có dạng `a + b·T`. Thu được:

- các **phương trình** `cur(i,j) = T` tại ô không đặt được góc;
- các **bất phương trình** `x_{ij} ≥ 0` tại vị trí đặt góc.

Phương trình có hệ số `T` khác 0 ⇒ `T` **duy nhất** (có thể rất lớn, như `T = 10` ở phản ví dụ). Nếu mọi hệ số bằng 0 ⇒ `T` tự do; vì số phép tăng theo `T`, chọn `T` **nhỏ nhất** thỏa các bất phương trình.

**Bước 3 — Dạng đóng của hệ số `T`.** Phần hệ số `T` của `x` chính là lời giải tham lam cho "lưới toàn 0, đích 1": `y_{ij} = 1` khi `i ≡ 0`, `j ≡ 0 (mod k)` (trong vùng đặt được), ngược lại `0`. Một ô `(a, b)` được phủ bởi phần này đúng 1 lần nếu khối chứa nó đặt được, ngược lại 0. Vì vậy:

- ô không đặt được góc và **không** được phủ bởi `y` ⇒ phương trình `cur₀(a,b) = T` (với `cur₀` là giá trị khi chạy tham lam ở `T = 0` cho phép số âm) ⇒ **`T = cur₀(a, b)`**;
- ô được phủ ⇒ phương trình `cur₀(a,b) = 0`.

**Bước 4 — Bất biến tổng.** Mỗi phép tăng tổng lưới `k²` ⇒ số phép `= (m·n·T - Σgrid) / k²`.

## 7. Các cách tiếp theo

- **Cách 2:** `check(T)` cho `T = max, max+1, …` lần lượt tới một giới hạn — đúng nếu đủ giới hạn, nhưng không biết dừng ở đâu (và cách chỉ thử `max`, `max+1` là **sai**).
- **Cách 3:** tham lam **affine theo `T`** (cặp `(a, b)`) + giải phương trình/bất phương trình — tổng quát, `O(mn)`.
- **Cách 4:** chạy tham lam **một lần với `T = 0`** (cho phép âm) + dạng đóng của hệ số `T` ⇒ tìm `T`, rồi kiểm tra bằng `check(T)` — `O(mn)`, lời giải nộp chính.

### 7.1. Cách 2 — `check(T)` và thử dần `T`

**Giải thích `check(T)`:** cửa sổ trượt theo cột như 3.2; tại ô đặt được góc, `x = T - cur` (âm ⇒ thất bại); tại ô không đặt được, `cur ≠ T` ⇒ thất bại.

In [6]:
def check(grid, k, T):
    m, n = len(grid), len(grid[0])
    col = [0] * n
    hist = []
    total = 0
    for i in range(m):
        if i >= k:
            old = hist[i - k]
            for c in range(n):
                col[c] -= old[c]
        ops_row = [0] * n
        window = 0
        row = grid[i]
        for j in range(n):
            window += col[j]
            if j >= k:
                window -= col[j - k]
            cur = row[j] + window
            if i <= m - k and j <= n - k:
                x = T - cur
                if x < 0:
                    return -1
                if x:
                    ops_row[j] = x
                    col[j] += x
                    window += x
                    total += x
            elif cur != T:
                return -1
        hist.append(ops_row)
    return total

def try_increasing(grid, k, limit=50):
    mx = max(max(r) for r in grid)
    for T in range(mx, mx + limit + 1):
        r = check(grid, k, T)
        if r != -1:
            return r
    return -1

for g, k, exp in examples:
    print(try_increasing(g, k), "kỳ vọng", exp)
print("phản ví dụ:", try_increasing([[5, 0, 5], [5, 0, 5]], 2), "| chỉ thử max, max+1:", try_increasing([[5, 0, 5], [5, 0, 5]], 2, limit=1))

2 kỳ vọng 2
4 kỳ vọng 4
phản ví dụ: 10 | chỉ thử max, max+1: -1


**Độ phức tạp Cách 2:** `O(mn)` mỗi `T`, nhân với số `T` thử. Không có chặn trên tốt cho `T` ⇒ không dùng được làm lời giải chính, nhưng `check` là viên gạch quan trọng.

### 7.2. Cách 3 — Tham lam affine theo `T`

**Giải thích từng đoạn:**

1. Mọi đại lượng là cặp `(a, b)` = `a + b·T`. Dùng hai bộ mảng `col_a/col_b`, `window_a/window_b`.
2. Tại ô đặt được góc: `x = (T - cur)` = `(-cur_a, 1 - cur_b)`; lưu vào danh sách bất phương trình `x ≥ 0`.
3. Tại ô không đặt được: phương trình `cur - T = (cur_a, cur_b - 1) = 0`.
4. Giải: gom các phương trình có `b ≠ 0` ⇒ một giá trị `T` duy nhất (phải nguyên và khớp mọi phương trình); phương trình `b = 0` cần `a = 0`. Nếu `T` tự do: `T_min = max(⌈-a/b⌉)` trên các bất phương trình có `b > 0`; kiểm tra các bất phương trình còn lại.
5. Đáp án `= Σ x = (mnT - Σgrid) / k²`.

In [7]:
def affine_solution(grid, k):
    m, n = len(grid), len(grid[0])
    col_a, col_b = [0] * n, [0] * n
    hist = []
    eqs, ineqs = [], []                  # eqs: a + bT = 0 ; ineqs: a + bT >= 0
    for i in range(m):
        if i >= k:
            oa, ob = hist[i - k]
            for c in range(n):
                col_a[c] -= oa[c]; col_b[c] -= ob[c]
        ra, rb = [0] * n, [0] * n
        wa = wb = 0
        for j in range(n):
            wa += col_a[j]; wb += col_b[j]
            if j >= k:
                wa -= col_a[j - k]; wb -= col_b[j - k]
            ca, cb = grid[i][j] + wa, wb
            if i <= m - k and j <= n - k:
                xa, xb = -ca, 1 - cb          # x = T - cur
                ineqs.append((xa, xb))
                ra[j], rb[j] = xa, xb
                col_a[j] += xa; col_b[j] += xb
                wa += xa; wb += xb
            else:
                eqs.append((ca, cb - 1))      # cur - T = 0
        hist.append((ra, rb))
    T = None
    for a, b in eqs:
        if b == 0:
            if a != 0:
                return -1
        else:
            if (-a) % b != 0:
                return -1
            t = -a // b
            if T is None:
                T = t
            elif T != t:
                return -1
    if T is None:                         # T tự do: lấy nhỏ nhất thỏa x >= 0
        lo = None
        for a, b in ineqs:
            if b > 0:
                need = (-a + b - 1) // b          # ceil(-a / b) với b > 0
                lo = need if lo is None else max(lo, need)
        T = lo if lo is not None else max(max(r) for r in grid)
    for a, b in ineqs:
        if a + b * T < 0:
            return -1
    return sum(a + b * T for a, b in ineqs)

for g, k, exp in examples:
    print(affine_solution(g, k), "kỳ vọng", exp)
print("phản ví dụ:", affine_solution([[5, 0, 5], [5, 0, 5]], 2))

2 kỳ vọng 2
4 kỳ vọng 4
phản ví dụ: 10


Ghi chú: `ceil(-a / b)` với `b > 0` viết gọn là `(-a + b - 1) // b` (phép chia lấy sàn của Python đúng cả với số âm).

In [8]:
class SolutionAffine:
    def minOperations(self, grid: List[List[int]], k: int) -> int:
        return affine_solution(grid, k)

print([SolutionAffine().minOperations(g, k) for g, k, _ in examples])

[2, 4]


**Độ phức tạp Cách 3:** `O(mn)` thời gian, `O(mn)` bộ nhớ (lưu lịch sử phép đặt của `k` hàng gần nhất — có thể giảm còn `O(kn)`) — hằng số gấp đôi vì mỗi đại lượng là một cặp.

### 7.3. Cách 4 — Một lượt với `T = 0` + dạng đóng (lời giải chính)

**Giải thích từng đoạn:**

1. Chạy tham lam với `T = 0` **cho phép số âm**: thu `cur₀` tại các ô không đặt được góc và `x₀` tại các vị trí đặt.
2. Với ô không đặt được `(a, b)`: nó được phủ bởi phần `y` ⇔ `a - a % k ≤ m - k` **và** `b - b % k ≤ n - k`. Nếu được phủ: cần `cur₀ = 0`. Nếu không: `T = cur₀` (mọi ô như vậy phải cho cùng một `T`).
3. Nếu chưa có `T` (khi `k | m` và `k | n`): `T = max(-x₀)` trên các vị trí `y = 1`.
4. Gọi `check(T)` để xác nhận (đơn giản, tránh sai sót); trả kết quả.

In [9]:
def one_pass_solution(grid, k):
    m, n = len(grid), len(grid[0])
    col = [0] * n
    hist = []
    T = None
    t_free_lower = None
    for i in range(m):
        if i >= k:
            old = hist[i - k]
            for c in range(n):
                col[c] -= old[c]
        ops_row = [0] * n
        window = 0
        row_cov = (i - i % k) <= m - k
        for j in range(n):
            window += col[j]
            if j >= k:
                window -= col[j - k]
            cur = grid[i][j] + window
            if i <= m - k and j <= n - k:
                x = -cur                                  # x0 = 0 - cur
                ops_row[j] = x; col[j] += x; window += x
                if i % k == 0 and j % k == 0:             # vị trí có y = 1: x = x0 + T >= 0
                    t_free_lower = -x if t_free_lower is None else max(t_free_lower, -x)
            else:
                covered = row_cov and (j - j % k) <= n - k
                if covered:
                    if cur != 0:
                        return -1
                else:
                    if T is None:
                        T = cur
                    elif T != cur:
                        return -1
        hist.append(ops_row)
    if T is None:
        T = t_free_lower
    return check(grid, k, T)

for g, k, exp in examples:
    print(one_pass_solution(g, k), "kỳ vọng", exp)
print("phản ví dụ:", one_pass_solution([[5, 0, 5], [5, 0, 5]], 2))

2 kỳ vọng 2
4 kỳ vọng 4
phản ví dụ: 10


**Độ phức tạp Cách 4:** hai lượt `O(mn)` (một lượt tìm `T`, một lượt `check`), bộ nhớ `O(mn)` cho lịch sử (có thể giảm `O(kn)`).

## 8. TRACE từng bước

Trace `check(T)` trên ví dụ 1 với `T = 5` (`grid = [[3,3,5],[3,3,5]]`, `k = 2`; vị trí đặt được: `(0,0)`, `(0,1)`).

| ô | đặt được? | cur | hành động | tổng |
|---|---|---|---|---|
| (0,0) | có | 3 | đặt `x = 2` | 2 |
| (0,1) | có | 3 + 2 = 5 | `x = 0` | 2 |
| (0,2) | không | 5 | cần `= 5` ✔ | 2 |
| (1,0) | không | 3 + 2 = 5 | ✔ | 2 |
| (1,1) | không | 5 | ✔ | 2 |
| (1,2) | không | 5 | ✔ | 2 |

In [10]:
def trace_check(grid, k, T):
    m, n = len(grid), len(grid[0])
    col = [0] * n; hist = []; total = 0
    for i in range(m):
        if i >= k:
            for c in range(n): col[c] -= hist[i - k][c]
        ops_row = [0] * n; window = 0
        for j in range(n):
            window += col[j]
            if j >= k: window -= col[j - k]
            cur = grid[i][j] + window
            if i <= m - k and j <= n - k:
                x = T - cur
                note = f"đặt x = {x}" if x >= 0 else f"x = {x} < 0 -> THẤT BẠI"
                print(f"({i},{j}) đặt được, cur={cur} -> {note}")
                if x < 0: return -1
                ops_row[j] = x; col[j] += x; window += x; total += x
            else:
                ok = cur == T
                print(f"({i},{j}) không đặt được, cur={cur} {'== T ✔' if ok else '!= T -> THẤT BẠI'}")
                if not ok: return -1
        hist.append(ops_row)
    print("tổng số phép:", total); return total

trace_check([[3, 3, 5], [3, 3, 5]], 2, 5)

(0,0) đặt được, cur=3 -> đặt x = 2
(0,1) đặt được, cur=5 -> đặt x = 0
(0,2) không đặt được, cur=5 == T ✔
(1,0) không đặt được, cur=5 == T ✔
(1,1) không đặt được, cur=5 == T ✔
(1,2) không đặt được, cur=5 == T ✔
tổng số phép: 2


2

In [11]:
# Trace phản ví dụ: T = 5, 6 thất bại; T = 10 thành công
for T in (5, 6, 10):
    print(f"--- T = {T} ---")
    trace_check([[5, 0, 5], [5, 0, 5]], 2, T)

--- T = 5 ---
(0,0) đặt được, cur=5 -> đặt x = 0
(0,1) đặt được, cur=0 -> đặt x = 5
(0,2) không đặt được, cur=10 != T -> THẤT BẠI
--- T = 6 ---
(0,0) đặt được, cur=5 -> đặt x = 1
(0,1) đặt được, cur=1 -> đặt x = 5
(0,2) không đặt được, cur=10 != T -> THẤT BẠI
--- T = 10 ---
(0,0) đặt được, cur=5 -> đặt x = 5
(0,1) đặt được, cur=5 -> đặt x = 5
(0,2) không đặt được, cur=10 == T ✔
(1,0) không đặt được, cur=10 == T ✔
(1,1) không đặt được, cur=10 == T ✔
(1,2) không đặt được, cur=10 == T ✔
tổng số phép: 10


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def minOperations(self, grid: List[List[int]], k: int) -> int:
        m, n = len(grid), len(grid[0])

        def run(T, allow_negative):
            """Quét tham lam. allow_negative=True: chế độ tìm T (T = 0)."""
            col = [0] * n
            hist = []
            total = 0
            Tfound = None
            lower = None
            for i in range(m):
                if i >= k:
                    old = hist[i - k]
                    for c in range(n):
                        if old[c]:
                            col[c] -= old[c]
                ops_row = [0] * n
                window = 0
                row = grid[i]
                can_i = i <= m - k
                row_cov = (i - i % k) <= m - k
                for j in range(n):
                    window += col[j]
                    if j >= k:
                        window -= col[j - k]
                    cur = row[j] + window
                    if can_i and j <= n - k:
                        x = T - cur
                        if x < 0 and not allow_negative:
                            return None
                        if x:
                            ops_row[j] = x
                            col[j] += x
                            window += x
                            total += x
                        if allow_negative and i % k == 0 and j % k == 0:
                            lower = -x if lower is None else max(lower, -x)
                    elif not allow_negative:
                        if cur != T:
                            return None
                    else:
                        if row_cov and (j - j % k) <= n - k:
                            if cur != 0:
                                return None
                        elif Tfound is None:
                            Tfound = cur
                        elif Tfound != cur:
                            return None
                hist.append(ops_row)
            return (Tfound, lower) if allow_negative else total

        found = run(0, True)
        if found is None:
            return -1
        T = found[0] if found[0] is not None else found[1]
        res = run(T, False)
        return -1 if res is None else res

sol = Solution()
for g, k, exp in examples:
    got = sol.minOperations(g, k)
    print(got, "✔" if got == exp else "✘")
print("phản ví dụ:", sol.minOperations([[5, 0, 5], [5, 0, 5]], 2))

2 ✔
4 ✔
phản ví dụ: 10


## 10. Vì sao thuật toán đúng

1. **Ép buộc theo thứ tự quét.** Đặt `x_{rc}` là số lần đặt góc tại `(r, c)`. Ô `(i, j)` nhận `Σ x_{rc}` với `i-k < r ≤ i`, `j-k < c ≤ j`. Trong các vị trí đó, chỉ `(i, j)` chưa được quyết định khi quét tới `(i, j)` (mọi `(r, c)` khác đứng trước theo thứ tự hàng). Nên với `T` cố định, `x_{ij}` **duy nhất** = `T - cur`; ô không đặt được thì phương trình phải tự thỏa. Vậy `check(T)` đúng và tập lời giải (nếu có) là duy nhất với mỗi `T`.
2. **Affine.** Các phép tính tham lam chỉ cộng/trừ ⇒ mọi `x_{ij}` và mọi `cur` là `a + b·T`. Điều kiện khả thi = các phương trình + `x ≥ 0`. Số phép `Σx = (mnT - Σgrid)/k²` tăng ngặt theo `T` ⇒ chọn `T` khả thi nhỏ nhất.
3. **Dạng đóng hệ số.** Phần hệ số `T` là lời giải tham lam cho lưới 0, đích 1: đặt 1 tại `(0,0)`, các ô trong khối đầu đã đủ; ô kế tiếp chưa đủ là `(0, k)`… ⇒ `y_{ij} = [i ≡ 0][j ≡ 0]` trong vùng đặt được; ô `(a,b)` được phủ đúng 1 lần nếu khối `(a - a%k, b - b%k)` đặt được, 0 lần nếu không. Ô không phủ và không đặt được ⇒ hệ số của `cur - T` là `-1` ⇒ `T = cur₀`.
4. **Doocs thiếu sót ở đâu?** Lập luận "nếu `max` và `max+1` đều không được thì không thể" sai, vì `T` có thể bị ép tới một giá trị lớn hơn nhiều (phản ví dụ: `T = 10 = max + 5`).

## 11. Phân tích Time / Space Complexity

Đặt `N = m·n ≤ 10^6`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O((B+1)^{(m-k+1)(n-k+1)} · N k²)` | `O(N)` |
| 2. Thử dần `T` + `check` | `O(N · số T thử)` | `O(N)` |
| 3. Tham lam affine | `O(N)` (hằng số ×2) | `O(N)` |
| 4. Một lượt `T = 0` + `check` | `O(N)` (2 lượt) | `O(N)` |

Với `N = 10^6`, hai lượt quét thuần Python mất vài giây — đây là ràng buộc nặng; có thể giảm bộ nhớ lịch sử xuống `O(kn)` bằng hàng đợi vòng.

## 12. Bộ test

In [13]:
def run_all(g, k, with_brute=False):
    res = [affine_solution(g, k), one_pass_solution(g, k), Solution().minOperations(g, k)]
    if with_brute:
        res.append(brute(g, k, B=6))
    return res

for g, k, exp in examples:
    assert all(v == exp for v in run_all(g, k, True)), run_all(g, k, True)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([[7]], 1, 0),                              # 1 ô
    ([[1, 2], [3, 4]], 2, -1),                  # k = m = n: chỉ một phép phủ hết -> cần sẵn bằng nhau
    ([[4, 4], [4, 4]], 2, 0),
    ([[5, 0, 5], [5, 0, 5]], 2, 10),            # phản ví dụ cho "chỉ thử max, max+1"
    ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], 2, 0),
    ([[0, 1, 0], [1, 2, 1], [0, 1, 0]], 2, -1),
]
for g, k, exp in cases:
    r = run_all(g, k, with_brute=True)
    assert all(v == exp for v in r), (g, k, r)
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 phản ví dụ được TẠO NGƯỢC: chọn ngẫu nhiên số lần đặt x >= 0, sinh lưới = T - phủ(x).
# Lưới như vậy luôn khả thi với T đã chọn; kiểm tra cách "chỉ thử max, max+1" bỏ sót bao nhiêu.
def doocs_style(grid, k):
    mx = max(max(r) for r in grid)
    for T in (mx, mx + 1):
        r = check(grid, k, T)
        if r != -1:
            return r
    return -1

rng = random.Random(3888)
missed = 0; total = 0
for _ in range(300):
    m, n = rng.randint(1, 5), rng.randint(1, 5); k = rng.randint(1, min(m, n))
    ops = [[rng.randint(0, 6) if i <= m - k and j <= n - k else 0 for j in range(n)] for i in range(m)]
    cov = coverage_by_window(m, n, k, ops)
    T = 10
    g = [[T - cov[i][j] for j in range(n)] for i in range(m)]
    ans = Solution().minOperations(g, k)
    assert ans != -1 and ans <= sum(map(sum, ops))
    total += 1; missed += doocs_style(g, k) == -1
print(f"lưới khả thi sinh ngẫu nhiên: {total}, cách chỉ thử max/max+1 trả -1 sai ở {missed} lưới")

lưới khả thi sinh ngẫu nhiên: 300, cách chỉ thử max/max+1 trả -1 sai ở 27 lưới


In [16]:
# 12.4 đối chiếu ngẫu nhiên với vét cạn (lưới nhỏ)
rng = random.Random(7)
for _ in range(150):
    m, n = rng.randint(1, 3), rng.randint(1, 3); k = rng.randint(1, min(m, n))
    if (m - k + 1) * (n - k + 1) > 4:
        continue
    g = [[rng.randint(0, 4) for _ in range(n)] for _ in range(m)]
    r = run_all(g, k, with_brute=True)
    assert len(set(r)) == 1, (g, k, r)
print("đối chiếu vét cạn: OK")
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8); k = rng.randint(1, min(m, n))
    ops = [[rng.randint(0, 5) if i <= m - k and j <= n - k else 0 for j in range(n)] for i in range(m)]
    cov = coverage_by_window(m, n, k, ops)
    g = [[rng.randint(20, 30) - cov[i][j] if rng.random() < 0.05 else 30 - cov[i][j] for j in range(n)] for i in range(m)]
    r = run_all(g, k)
    assert len(set(r)) == 1, (g, k, r)
print("đối chiếu 3 cách trên 300 lưới vừa: OK")

đối chiếu vét cạn: OK
đối chiếu 3 cách trên 300 lưới vừa: OK


In [17]:
# 12.5 test lớn (300 x 300 để notebook chạy nhanh; ràng buộc thật 1000 x 1000)
rng = random.Random(9)
m = n = 300; k = 7
ops = [[rng.randint(0, 3) if i <= m - k and j <= n - k else 0 for j in range(n)] for i in range(m)]
cov = coverage_by_window(m, n, k, ops)
g = [[100 - cov[i][j] for j in range(n)] for i in range(m)]
t0 = time.time(); a = Solution().minOperations(g, k); t1 = time.time()
S = sum(map(sum, g))
print("đáp án:", a, "| khớp affine:", a == affine_solution(g, k), "| thời gian:", round(t1 - t0, 2), "s")
print("bất biến tổng: (mnT - S)/k^2 với T = 100 ->", (m * n * 100 - S) // (k * k), "| tổng ops đã sinh:", sum(map(sum, ops)))

đáp án: 129306 | khớp affine: True | thời gian: 0.04 s
bất biến tổng: (mnT - S)/k^2 với T = 100 -> 129306 | tổng ops đã sinh: 129306


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ thử `T = max(grid)` (hoặc `max` và `max + 1`)

Đã thấy ở 12.3: nhiều lưới khả thi chỉ với `T` lớn hơn nhiều. **Sửa:** suy ra `T` từ phương trình (Cách 3/4).

In [18]:
g = [[5, 0, 5], [5, 0, 5]]
print("chỉ thử max:", check(g, 2, 5), "| max+1:", check(g, 2, 6), "| đúng:", Solution().minOperations(g, 2))

chỉ thử max: -1 | max+1: -1 | đúng: 10


### Lỗi 2 — Cộng `k × k` bằng vòng lặp lồng nhau

Mỗi lần đặt góc mà duyệt `k²` ô ⇒ `O(mn k²)` = `10^{12}` với `k = 1000`. **Sửa:** mảng hiệu 2D hoặc cửa sổ trượt cột.

In [19]:
def check_slow(grid, k, T):
    m, n = len(grid), len(grid[0]); g = [r[:] for r in grid]; total = 0
    for i in range(m):
        for j in range(n):
            if i <= m - k and j <= n - k:
                x = T - g[i][j]
                if x < 0: return -1
                for a in range(i, i + k):           # O(k^2) mỗi lần
                    for b in range(j, j + k):
                        g[a][b] += x
                total += x
            elif g[i][j] != T:
                return -1
    return total

g = [[rng.randint(0, 3) for _ in range(40)] for _ in range(40)]
t0 = time.time(); s1 = check_slow(g, 20, 50); t1 = time.time(); s2 = check(g, 20, 50); t2 = time.time()
print("cùng kết quả:", s1 == s2, "| chậm hơn:", (t1 - t0) > (t2 - t1))

cùng kết quả: True | chậm hơn: True


### Lỗi 3 — Quên kiểm tra ô ở **mép phải/dưới** (không đặt được góc)

Nếu chỉ đặt góc khi `cur < T` mà không kiểm tra các ô ngoài vùng đặt được, ta có thể trả về số phép dù lưới chưa bằng nhau.

In [20]:
def wrong_no_edge(grid, k, T):
    m, n = len(grid), len(grid[0]); g = [r[:] for r in grid]; total = 0
    for i in range(m - k + 1):
        for j in range(n - k + 1):
            x = T - g[i][j]
            if x < 0: return -1
            for a in range(i, i + k):
                for b in range(j, j + k): g[a][b] += x
            total += x
    return total                                   # SAI: không kiểm tra các ô còn lại

g = [[1, 2], [3, 4]]
print("sai:", wrong_no_edge(g, 2, 4), "| đúng:", Solution().minOperations(g, 2))

sai: 3 | đúng: -1


### Lỗi 4 — Giải `T` bằng phép chia thực (float)

Ở Cách 3, `T = -a / b` bằng số thực có thể ra `9.999999` hoặc chấp nhận `T` không nguyên. **Sửa:** kiểm tra `(-a) % b == 0` rồi dùng `//`.

In [21]:
a, b = -7, 2
print("float:", -a / b, "| kiểm tra chia hết:", (-a) % b == 0, "-> vô nghiệm nguyên")

float: 3.5 | kiểm tra chia hết: False -> vô nghiệm nguyên


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mỗi thao tác cộng vào một cửa sổ cố định kích thước"** + "làm mọi phần tử bằng nhau / bằng 0" ⇒ **quét tham lam từ đầu**: phần tử đầu tiên chỉ có một cách sửa (*Minimum Number of K Consecutive Bit Flips* LC 995, *Apply Operations to Make All Array Elements Equal to Zero* LC 2772).
- Phép cộng cửa sổ ⇒ **mảng hiệu** (1D/2D) để mỗi thao tác `O(1)`.
- Khi đích `T` chưa biết ⇒ coi `T` là **ẩn**, mọi đại lượng affine ⇒ giải phương trình thay vì thử.
- **Bất biến tổng**: số phép = (tổng cuối − tổng đầu) / kích thước cửa sổ.

## 15. Biến thể của bài

1. **1D**: mảng độ dài `n`, mỗi phép cộng 1 vào đoạn độ dài `k` — cùng tư duy (LC 2772 là phiên bản trừ về 0).
2. **Cho phép cả tăng và giảm**: mọi `x` được âm ⇒ chỉ còn ràng buộc phương trình; số phép = `Σ |x|`.
3. **Đích cố định** (ví dụ phải bằng 0, chỉ được giảm) ⇒ chỉ cần một lần `check`.
4. **Ma trận con `p × q`** (không vuông): thay `k` bằng `p` theo hàng, `q` theo cột.

In [22]:
# Biến thể 1 (1D): mảng a, mỗi phép +1 vào đoạn độ dài k; tối thiểu số phép để mọi phần tử bằng nhau
def one_dim_direct(a, k):
    # coi như lưới 1 x n với ma trận con 1 x k: tự viết lại check affine
    n = len(a)
    best = None
    for T in range(max(a), max(a) + 60):
        add = [0] * (n + 1); cur_add = 0; total = 0; ok = True
        for i in range(n):
            cur_add += add[i]
            cur = a[i] + cur_add
            if i <= n - k:
                x = T - cur
                if x < 0: ok = False; break
                cur_add += x; add[i + k] -= x; total += x
            elif cur != T:
                ok = False; break
        if ok:
            best = total; break
    return -1 if best is None else best

print("1D [5,0,5], k=2:", one_dim_direct([5, 0, 5], 2), "| [3,1,2,4], k=2:", one_dim_direct([3, 1, 2, 4], 2))
# lưới 1 x n không dùng được với k >= 2 trong bài gốc (k <= min(m, n)), nên 1D là bài riêng

1D [5,0,5], k=2: 10 | [3,1,2,4], k=2: 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi số lần đặt | mũ | `O(N)` | Kiểm thử lưới tí hon |
| 2. Thử dần `T` | `check(T)` từ `max` trở lên | `O(N · #T)` | `O(N)` | Khi biết `T` nằm trong khoảng nhỏ |
| 3. Affine theo `T` | Cặp `(a, b)`, giải hệ | `O(N)` | `O(N)` | Tổng quát, không cần dạng đóng |
| 4. `T = 0` + dạng đóng | Tìm `T` một lượt, rồi `check` | `O(N)` | `O(N)` | **Lời giải chuẩn**, code gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `grid = [[2, 2], [2, 2]], k = 1` → ?

**Bài 2 (tính tay).** `grid = [[1, 3, 2]], k = 1`… và `grid = [[4, 1, 3], [4, 1, 3]], k = 2` — tính `T` và đáp án.

**Bài 3 (điền TODO).** Viết `coverage_diff2d(m, n, k, ops)` dùng **mảng hiệu 2D** thay cho cửa sổ trượt.

**Bài 4 (tự viết lại).** Viết `check(T)` dùng mảng hiệu 2D (không dùng cửa sổ trượt).

**Bài 5 (mini challenge).** Biến thể cho phép **tăng hoặc giảm** (mỗi phép `±1` trên `k × k`). Tìm số phép tối thiểu (gợi ý: mọi `x` là affine theo `T`; số phép `Σ |a + bT|` là hàm lồi theo `T` ⇒ thử `T` quanh trung vị).

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def coverage_diff2d(m, n, k, ops):
    # TODO: D[i][j] += x, D[i+k][j] -= x, D[i][j+k] -= x, D[i+k][j+k] += x; rồi tiền tố 2D
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def min_ops_plus_minus(grid, k):
    # TODO: chạy tham lam affine; nếu T bị ép thì tính Σ|x|; nếu T tự do thì tối thiểu hóa Σ|a + bT|
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đã bằng nhau. (2) `T = 2`. (3) Đáp án `0`.

**Bài 2.** (1) `k = 1`: `T = max = 3`, đáp án `2 + 0 + 1 = 3`. (2) Lưới thứ hai: `x₀ = T - 4`, `x₁ = T - 3`, cột giữa `1 + x₀ + x₁ = T` ⇒ `T = 6`. (3) Đáp án `2 + 3 = 5`.

**Bài 3.** (1) Kích thước `D` là `(m+1) × (n+1)`. (2) Tiền tố `P[i][j] = D[i][j] + P[i-1][j] + P[i][j-1] - P[i-1][j-1]`. (3) So với `coverage_by_window`.

**Bài 4.** (1) Tính tiền tố ngay khi quét. (2) Khi đặt `x` tại `(i, j)` thì cập nhật `D` và cộng `x` vào giá trị hiện tại. (3) So với `check`.

**Bài 5.** (1) Bỏ điều kiện `x ≥ 0`. (2) Nếu `T` bị ép: `Σ|x|`. (3) Nếu tự do: `f(T) = Σ|a + T·b|` với `b ∈ {0, 1}` ⇒ trung vị của `-a` trên các `b = 1`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().minOperations([[2, 2], [2, 2]], 1) == 0
assert Solution().minOperations([[1, 3, 2]], 1) == 3
assert Solution().minOperations([[4, 1, 3], [4, 1, 3]], 2) == 5 == brute([[4, 1, 3], [4, 1, 3]], 2, B=6)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3
def coverage_diff2d_sol(m, n, k, ops):
    D = [[0] * (n + k + 1) for _ in range(m + k + 1)]
    for i in range(m):
        for j in range(n):
            x = ops[i][j]
            if x:
                D[i][j] += x; D[i + k][j] -= x; D[i][j + k] -= x; D[i + k][j + k] += x
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = D[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return [row[1:] for row in P[1:]]

rng = random.Random(3)
for _ in range(100):
    m, n = rng.randint(1, 7), rng.randint(1, 7); k = rng.randint(1, min(m, n))
    ops = [[rng.randint(0, 4) if i <= m - k and j <= n - k else 0 for j in range(n)] for i in range(m)]
    assert coverage_diff2d_sol(m, n, k, ops) == coverage_by_window(m, n, k, ops)
print("Bài 3 OK")

Bài 3 OK


In [27]:
# Bài 4
def check_diff2d(grid, k, T):
    m, n = len(grid), len(grid[0])
    D = [[0] * (n + k + 1) for _ in range(m + k + 1)]
    P = [[0] * (n + 1) for _ in range(m + 1)]
    total = 0
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = D[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
            cur = grid[i][j] + P[i + 1][j + 1]
            if i <= m - k and j <= n - k:
                x = T - cur
                if x < 0: return -1
                if x:
                    D[i][j] += x; D[i + k][j] -= x; D[i][j + k] -= x; D[i + k][j + k] += x
                    P[i + 1][j + 1] += x; total += x
            elif cur != T:
                return -1
    return total

rng = random.Random(4)
for _ in range(200):
    m, n = rng.randint(1, 6), rng.randint(1, 6); k = rng.randint(1, min(m, n))
    g = [[rng.randint(0, 5) for _ in range(n)] for _ in range(m)]
    for T in range(5, 12):
        assert check_diff2d(g, k, T) == check(g, k, T)
print("Bài 4 OK")

Bài 4 OK


In [28]:
# Bài 5: tăng hoặc giảm
def min_ops_pm(grid, k):
    m, n = len(grid), len(grid[0])
    # thu các x dạng affine (a, b) giống Cách 3 nhưng không ràng buộc dấu
    col_a, col_b = [0] * n, [0] * n; hist = []; eqs = []; xs = []
    for i in range(m):
        if i >= k:
            oa, ob = hist[i - k]
            for c in range(n): col_a[c] -= oa[c]; col_b[c] -= ob[c]
        ra, rb = [0] * n, [0] * n; wa = wb = 0
        for j in range(n):
            wa += col_a[j]; wb += col_b[j]
            if j >= k: wa -= col_a[j - k]; wb -= col_b[j - k]
            ca, cb = grid[i][j] + wa, wb
            if i <= m - k and j <= n - k:
                xa, xb = -ca, 1 - cb; xs.append((xa, xb)); ra[j], rb[j] = xa, xb
                col_a[j] += xa; col_b[j] += xb; wa += xa; wb += xb
            else:
                eqs.append((ca, cb - 1))
        hist.append((ra, rb))
    T = None
    for a, b in eqs:
        if b == 0:
            if a: return -1
        else:
            if (-a) % b: return -1
            t = -a // b
            if T is not None and T != t: return -1
            T = t
    if T is None:
        pts = sorted(-a // b for a, b in xs if b != 0) or [0]
        cands = {pts[len(pts) // 2], pts[(len(pts) - 1) // 2]}
        return min(sum(abs(a + b * t) for a, b in xs) for t in cands)
    return sum(abs(a + b * T) for a, b in xs)

def brute_pm(grid, k, B=4):
    m, n = len(grid), len(grid[0]); pos = [(i, j) for i in range(m - k + 1) for j in range(n - k + 1)]; best = None
    for xs in product(range(-B, B + 1), repeat=len(pos)):
        g = [r[:] for r in grid]
        for (i, j), x in zip(pos, xs):
            for a in range(i, i + k):
                for b in range(j, j + k): g[a][b] += x
        if len({v for r in g for v in r}) == 1:
            s = sum(abs(x) for x in xs); best = s if best is None else min(best, s)
    return -1 if best is None else best

rng = random.Random(5)
for _ in range(120):
    m, n = rng.randint(1, 3), rng.randint(1, 3); k = rng.randint(1, min(m, n))
    if (m - k + 1) * (n - k + 1) > 4: continue
    g = [[rng.randint(0, 3) for _ in range(n)] for _ in range(m)]
    assert min_ops_pm(g, k) == brute_pm(g, k), (g, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# check(T): quét hàng, cửa sổ trượt theo cột
for i: (trừ các phép đặt ở hàng i-k khỏi col)
    window = 0
    for j:
        window += col[j] - (col[j-k] if j >= k else 0)
        cur = grid[i][j] + window
        if (i, j) đặt được góc: x = T - cur; x < 0 -> -1; col[j] += x; window += x
        elif cur != T: -1
# tìm T: chạy với T = 0 (cho phép âm);
#   ô không đặt được & không được phủ bởi y -> T = cur0;  được phủ -> cur0 phải = 0
#   nếu không có -> T = max(-x0) trên các (i%k==0, j%k==0)
```

**Checklist:**

- [ ] Không chỉ thử `T = max` / `max + 1` — suy ra `T` từ phương trình?
- [ ] Cộng `k × k` bằng mảng hiệu / cửa sổ trượt (không `O(k²)` mỗi lần)?
- [ ] Kiểm tra **mọi** ô ngoài vùng đặt được góc?
- [ ] Giải `T` bằng số nguyên (kiểm tra chia hết)?
- [ ] Bất biến: đáp án `= (m·n·T − Σgrid) / k²` (dùng để tự kiểm)?
- [ ] (Premium) Đã đối chiếu đề/skeleton gốc?